In [15]:
import os
import torch
import torch.nn as nn
import numpy as np
import pretty_midi
from collections import OrderedDict

# --- 1. SETTINGS ---
# Wir nutzen GPU 1, wie besprochen. Falls es hakt, ändere es auf "cpu" zum Testen.
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Initialisierung auf: {device} ---")

# --- 2. MODEL DEFINITIONS ---
class MusicLSTM(nn.Module):
    def __init__(self, input_size=4, hidden_size=256, num_layers=2):
        super(MusicLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.intermediate = nn.Linear(hidden_size, 128)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(128, 4)
        
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.relu(self.intermediate(out))
        return self.fc(out)

class MusicGRU(nn.Module):
    def __init__(self, input_size=4, hidden_size=512):
        super(MusicGRU, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=3, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 4)
        
    def forward(self, x):
        out, _ = self.gru(x)
        return torch.sigmoid(self.fc(out[:, -1, :]))

class MusicTransformer(nn.Module):
    def __init__(self, input_size=4, d_model=128, nhead=8, num_layers=4, window_size=64):
        super(MusicTransformer, self).__init__()
        self.embedding = nn.Linear(input_size, d_model)
        
        self.pos_embedding = nn.Parameter(torch.zeros(1, window_size, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=512,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc_hidden = nn.Linear(d_model, 64)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(64, 4)

    def forward(self, x):
        
        x = self.embedding(x) + self.pos_embedding
        
        out = self.transformer(x)
        
        out = out[:, -1, :]
        out = self.relu(self.fc_hidden(out))
        return self.fc_out(out)

# --- 3. HELPER FUNCTIONS ---

def load_clean_model(model_class, path, device):
    if not os.path.exists(path):
        print(f"⚠️ Datei nicht gefunden: {path}")
        return None
    
    model = model_class().to(device)
    state_dict = torch.load(path, map_location=device)
    
    # Säuberung des module-Präfixes von DataParallel
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k[7:] if k.startswith('module.') else k
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict)
    model.eval()
    print(f"✅ {path} erfolgreich geladen.")
    return model

def generate_and_save(model, model_name, device, gen_length=400):
    print(f"🎵 Starte Generierung für {model_name}...")
    
    # Dummy Seed (64 Schritte Stille/Neutral), falls kein Dataset geladen ist
    input_seq = torch.zeros((1, 64, 4)).to(device)
    input_seq[:, :, 0] = 0.5  # Mittlere Tonlage als Startpunkt
    
    generated_notes = []
    
    with torch.no_grad():
        for i in range(gen_length):
            output = model(input_seq)
            
            # Varianz/Rauschen hinzufügen
            noise = torch.randn_like(output) * 0.1
            output = torch.clamp(output + noise, 0, 1)
            
            note_data = output.cpu().numpy()[0]
            generated_notes.append(note_data)
            
            # Update Window
            new_input = output.unsqueeze(1)
            input_seq = torch.cat((input_seq[:, 1:, :], new_input), dim=1)
            
            if i % 100 == 0:
                print(f"   Progress {model_name}: {i}/{gen_length}")

    # MIDI Export
    os.makedirs("output_midi", exist_ok=True)
    pm = pretty_midi.PrettyMIDI()
    piano = pretty_midi.Instrument(program=0)
    
    curr_t = 0
    for n in generated_notes:
        pitch = int(n[0] * 127)
        step = max(0, n[1] * 1.0)
        dur = max(0.1, n[2] * 0.5)
        vel = int(n[3] * 127)
        
        if pitch > 0:
            start = curr_t + step
            end = start + dur
            piano.notes.append(pretty_midi.Note(velocity=vel, pitch=pitch, start=start, end=end))
            curr_t = start
            
    pm.instruments.append(piano)
    save_path = f"output_midi/variance_midi_{model_name}.mid"
    pm.write(save_path)
    print(f"💾 Gespeichert: {save_path}")

# --- 4. EXECUTION ---

configs = [
    (MusicLSTM, "LSTM", "midi_models/LSTM_v2.pth"),
    (MusicGRU, "GRU", "midi_models/GRU_v2.pth"),
    (MusicTransformer, "Transformer", "midi_models/Transformer_v2.pth")
]

for m_class, name, path in configs:
    m = load_clean_model(m_class, path, device)
    if m:
        generate_and_save(m, name, device)

print("\n--- Alle Aufgaben abgeschlossen! ---")

--- Initialisierung auf: cuda ---
✅ midi_models/LSTM_v1.pth erfolgreich geladen.
🎵 Starte Generierung für LSTM...
   Progress LSTM: 0/400
   Progress LSTM: 100/400


/tmp/ipykernel_3508535/1108059628.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)


   Progress LSTM: 200/400
   Progress LSTM: 300/400
💾 Gespeichert: output_midi/variance_midi_LSTM.mid


RuntimeError: Error(s) in loading state_dict for MusicGRU:
	Unexpected key(s) in state_dict: "gru.weight_ih_l2", "gru.weight_hh_l2", "gru.bias_ih_l2", "gru.bias_hh_l2". 
	size mismatch for gru.weight_ih_l0: copying a param with shape torch.Size([1536, 4]) from checkpoint, the shape in current model is torch.Size([768, 4]).
	size mismatch for gru.weight_hh_l0: copying a param with shape torch.Size([1536, 512]) from checkpoint, the shape in current model is torch.Size([768, 256]).
	size mismatch for gru.bias_ih_l0: copying a param with shape torch.Size([1536]) from checkpoint, the shape in current model is torch.Size([768]).
	size mismatch for gru.bias_hh_l0: copying a param with shape torch.Size([1536]) from checkpoint, the shape in current model is torch.Size([768]).
	size mismatch for gru.weight_ih_l1: copying a param with shape torch.Size([1536, 512]) from checkpoint, the shape in current model is torch.Size([768, 256]).
	size mismatch for gru.weight_hh_l1: copying a param with shape